# Team 9 — Multi-Hazard Weather Impact Monitoring

## Workshop 1: Source Profiling and Data Contracts

**Status: partial.** First samples and profile sections are present for all four sources. The full notebook still requires a clean top-to-bottom execution, joint review, and shared regional/cross-source decisions.

**Course:** Data Analysis Programming — 2026-II  
**Institution:** Universidad Distrital Francisco Jose de Caldas  
**Team:** 9  
**Members:** Kevin Rincon and Pablo Garzon  
**Candidate study area:** Oklahoma, United States  
**Status:** Candidate region; final approval pending four-source validation

### Purpose

This notebook profiles preserved, versioned evidence for the four assigned weather sources. The current version implements the NOAA Storm Events and NWS Alerts sections owned by Kevin. Pablo must add GHCN-Daily and NWS Station Observations using the same evidence and profiling standard.

The notebook does not modify raw evidence, perform live API calls, implement production ingestion, or claim that the preliminary Compound Hazard Index has been validated.

## 1. Setup and reproducibility

Source acquisition is handled by scripts under `scripts/workshop_1/`. This notebook reads only preserved local evidence so that rerunning it does not produce different API responses.

The workflow follows the course pattern: load, profile, describe, visualize, identify quality constraints, and document. Raw evidence is never cleaned or overwritten in place.

In [ ]:
# Run only in a clean Google Colab environment.
# %pip install -q pandas numpy matplotlib seaborn pyyaml


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid")

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Executed at UTC:", datetime.now(timezone.utc).isoformat())

In [ ]:
def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "PROJECT_CONTEXT.md").exists():
            return candidate
    raise FileNotFoundError(
        "Repository root not found. Run this notebook from inside the cloned project."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
SAMPLES_ROOT = PROJECT_ROOT / "workshop_1" / "samples"

NOAA_SAMPLE_PATH = SAMPLES_ROOT / "noaa_storm_events" / "details_sample.csv"
NOAA_METADATA_PATH = SAMPLES_ROOT / "noaa_storm_events" / "source_metadata.yaml"
NWS_POLL_1_PATH = SAMPLES_ROOT / "nws_alerts" / "poll_1.geojson"
NWS_POLL_2_PATH = SAMPLES_ROOT / "nws_alerts" / "poll_2.geojson"
NWS_POLL_1_METADATA_PATH = SAMPLES_ROOT / "nws_alerts" / "poll_1_metadata.yaml"
NWS_POLL_2_METADATA_PATH = SAMPLES_ROOT / "nws_alerts" / "poll_2_metadata.yaml"
NWS_RECENT_PAGE_1_PATH = SAMPLES_ROOT / "nws_alerts" / "recent_7d_page_1.geojson"
NWS_RECENT_PAGE_2_PATH = SAMPLES_ROOT / "nws_alerts" / "recent_7d_page_2.geojson"
NWS_RECENT_METADATA_PATH = SAMPLES_ROOT / "nws_alerts" / "recent_7d_metadata.yaml"

print("Project root:", PROJECT_ROOT)
GHCN_SAMPLE_PATH = SAMPLES_ROOT / "ghcn_daily_usw00003017_2025.csv"
GHCN_METADATA_PATH = SAMPLES_ROOT / "ghcn_daily_usw00003017_2025.metadata.json"
GHCN_PILOT_PATH = SAMPLES_ROOT / "ghcn_daily_denver_2025-06_to_09.csv"
GHCN_PILOT_METADATA_PATH = SAMPLES_ROOT / "ghcn_daily_denver_2025-06_to_09.metadata.json"
STATION_OBSERVATIONS_PATH = SAMPLES_ROOT / "nws_kden_observations_poll1.geojson"
STATION_OBSERVATIONS_POLL2_PATH = SAMPLES_ROOT / "nws_kden_observations_poll2.geojson"
STATION_OBSERVATIONS_POLL2_METADATA_PATH = SAMPLES_ROOT / "nws_kden_observations_poll2.metadata.json"
STATION_OBSERVATIONS_METADATA_PATH = SAMPLES_ROOT / "nws_kden_observations_poll1.metadata.json"

### 1.1 Evidence inventory and integrity

Every required artifact must exist and match its registered SHA-256 checksum before profiling begins. A missing or changed file stops the notebook rather than silently producing results from different evidence.

In [ ]:
EVIDENCE_PATHS = {
    "NOAA sample": NOAA_SAMPLE_PATH,
    "NOAA metadata": NOAA_METADATA_PATH,
    "GHCN-Daily 2025 sample": GHCN_SAMPLE_PATH,
    "GHCN-Daily 2025 metadata": GHCN_METADATA_PATH,
    "GHCN-Daily exploratory pilot": GHCN_PILOT_PATH,
    "GHCN-Daily exploratory metadata": GHCN_PILOT_METADATA_PATH,
    "NWS station observations poll 1": STATION_OBSERVATIONS_PATH,
    "NWS station observations metadata": STATION_OBSERVATIONS_METADATA_PATH,
    "NWS station observations poll 2": STATION_OBSERVATIONS_POLL2_PATH,
    "NWS station observations poll 2 metadata": STATION_OBSERVATIONS_POLL2_METADATA_PATH,
    "NWS poll 1": NWS_POLL_1_PATH,
    "NWS poll 2": NWS_POLL_2_PATH,
    "NWS poll 1 metadata": NWS_POLL_1_METADATA_PATH,
    "NWS poll 2 metadata": NWS_POLL_2_METADATA_PATH,
    "NWS recent page 1": NWS_RECENT_PAGE_1_PATH,
    "NWS recent page 2": NWS_RECENT_PAGE_2_PATH,
    "NWS recent metadata": NWS_RECENT_METADATA_PATH,
}

evidence_inventory = pd.DataFrame([
    {
        "artifact": name,
        "path": str(path.relative_to(PROJECT_ROOT)),
        "exists": path.exists(),
        "bytes": path.stat().st_size if path.exists() else None,
    }
    for name, path in EVIDENCE_PATHS.items()
])
display(evidence_inventory)

missing_files = evidence_inventory.loc[~evidence_inventory["exists"], "artifact"].tolist()
if missing_files:
    raise FileNotFoundError(f"Required evidence is missing: {missing_files}")

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest().upper()


def load_yaml(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as file:
        return yaml.safe_load(file)


def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


noaa_metadata = load_yaml(NOAA_METADATA_PATH)
nws_poll_1_metadata = load_yaml(NWS_POLL_1_METADATA_PATH)
nws_poll_2_metadata = load_yaml(NWS_POLL_2_METADATA_PATH)
nws_recent_metadata = load_yaml(NWS_RECENT_METADATA_PATH)
ghcn_metadata = load_json(GHCN_METADATA_PATH)
ghcn_pilot_metadata = load_json(GHCN_PILOT_METADATA_PATH)
station_observations_metadata = load_json(STATION_OBSERVATIONS_METADATA_PATH)
station_observations_poll2_metadata = load_json(STATION_OBSERVATIONS_POLL2_METADATA_PATH)

integrity_checks = pd.DataFrame([
    {
        "artifact": "GHCN-Daily sample",
        "expected_sha256": ghcn_metadata["response"]["sha256"].upper(),
        "observed_sha256": sha256_file(GHCN_SAMPLE_PATH),
    },
    {
        "artifact": "GHCN-Daily exploratory pilot",
        "expected_sha256": ghcn_pilot_metadata["response"]["sha256"].upper(),
        "observed_sha256": sha256_file(GHCN_PILOT_PATH),
    },
    {
        "artifact": "NWS station observations poll 1",
        "expected_sha256": station_observations_metadata["response"]["sha256"].upper(),
        "observed_sha256": sha256_file(STATION_OBSERVATIONS_PATH),
    },
    {
        "artifact": "NWS station observations poll 2",
        "expected_sha256": station_observations_poll2_metadata["response"]["sha256"].upper(),
        "observed_sha256": sha256_file(STATION_OBSERVATIONS_POLL2_PATH),
    },
    {
        "artifact": "NOAA sample",
        "expected_sha256": noaa_metadata["sample"]["sha256"],
        "observed_sha256": sha256_file(NOAA_SAMPLE_PATH),
    },
    {
        "artifact": "NWS poll 1",
        "expected_sha256": nws_poll_1_metadata["evidence"]["sha256"],
        "observed_sha256": sha256_file(NWS_POLL_1_PATH),
    },
    {
        "artifact": "NWS poll 2",
        "expected_sha256": nws_poll_2_metadata["evidence"]["sha256"],
        "observed_sha256": sha256_file(NWS_POLL_2_PATH),
    },
    {
        "artifact": "NWS recent page 1",
        "expected_sha256": nws_recent_metadata["pagination"]["page_1"]["sha256"],
        "observed_sha256": sha256_file(NWS_RECENT_PAGE_1_PATH),
    },
    {
        "artifact": "NWS recent page 2",
        "expected_sha256": nws_recent_metadata["pagination"]["page_2"]["sha256"],
        "observed_sha256": sha256_file(NWS_RECENT_PAGE_2_PATH),
    },
])
integrity_checks["matches"] = (
    integrity_checks["expected_sha256"] == integrity_checks["observed_sha256"]
)
display(integrity_checks)
assert integrity_checks["matches"].all(), "At least one evidence checksum does not match."

### 1.2 Reusable profiling helpers

In [ ]:
def stable_value(value):
    if isinstance(value, (list, dict)):
        return json.dumps(value, sort_keys=True, default=str)
    return value


def safe_nunique(series: pd.Series) -> int:
    return series.dropna().map(stable_value).nunique()


def dataframe_profile(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for column in df.columns:
        series = df[column]
        rows.append({
            "column": column,
            "dtype": str(series.dtype),
            "non_null": int(series.notna().sum()),
            "null_count": int(series.isna().sum()),
            "null_pct": round(float(series.isna().mean() * 100), 2),
            "distinct": int(safe_nunique(series)),
        })
    return (
        pd.DataFrame(rows)
        .sort_values(["null_pct", "distinct"], ascending=[False, True])
        .reset_index(drop=True)
    )


def load_geojson(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as file:
        payload = json.load(file)
    if payload.get("type") != "FeatureCollection":
        raise ValueError(f"{path.name} is not a GeoJSON FeatureCollection.")
    return payload

## 2. NOAA Storm Events

The evidence is a deterministic 250-record subset of the official 2024 Storm Events details archive. It preserves all 51 source columns and represents 14 selected hazard types for Oklahoma. The untouched compressed archive remains in ignored raw storage.

In [ ]:
noaa_raw = pd.read_csv(NOAA_SAMPLE_PATH, dtype="string", keep_default_na=True)
noaa = noaa_raw.copy()

print("Rows:", len(noaa_raw))
print("Columns:", noaa_raw.shape[1])
display(noaa_raw.head())

assert len(noaa_raw) >= 200, "NOAA evidence does not meet the Workshop 1 record threshold."

### 2.1 Schema, missingness, and identifiers

In [ ]:
noaa_profile = dataframe_profile(noaa_raw)
display(noaa_profile)

noaa_id_summary = pd.DataFrame([{
    "rows": len(noaa_raw),
    "distinct_event_ids": noaa_raw["EVENT_ID"].nunique(dropna=True),
    "missing_event_ids": noaa_raw["EVENT_ID"].isna().sum(),
    "duplicated_event_ids": noaa_raw["EVENT_ID"].duplicated(keep=False).sum(),
    "exact_duplicate_rows": noaa_raw.duplicated().sum(),
}])
display(noaa_id_summary)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
null_plot = noaa_profile.sort_values("null_pct", ascending=True).tail(25)
ax.barh(null_plot["column"], null_plot["null_pct"], color="firebrick", alpha=0.75)
ax.set(title="NOAA Storm Events — Highest Missingness", xlabel="Null values (%)", ylabel="Column")
plt.tight_layout()
plt.show()

### 2.2 Temporal and geographic validation

Storm Events timestamps are parsed as source-local times. `CZ_TIMEZONE` must be interpreted before production UTC conversion; this notebook does not silently label local values as UTC.

In [ ]:
noaa["BEGIN_PARSED_LOCAL"] = pd.to_datetime(
    noaa["BEGIN_DATE_TIME"], format="%d-%b-%y %H:%M:%S", errors="coerce"
)
noaa["END_PARSED_LOCAL"] = pd.to_datetime(
    noaa["END_DATE_TIME"], format="%d-%b-%y %H:%M:%S", errors="coerce"
)
noaa["DURATION_HOURS"] = (
    noaa["END_PARSED_LOCAL"] - noaa["BEGIN_PARSED_LOCAL"]
).dt.total_seconds() / 3600

for column in ["BEGIN_LAT", "BEGIN_LON", "END_LAT", "END_LON", "MAGNITUDE"]:
    noaa[column + "_NUM"] = pd.to_numeric(noaa[column], errors="coerce")

temporal_summary = pd.DataFrame([{
    "begin_parse_failures": noaa["BEGIN_PARSED_LOCAL"].isna().sum(),
    "end_parse_failures": noaa["END_PARSED_LOCAL"].isna().sum(),
    "minimum_begin_local": noaa["BEGIN_PARSED_LOCAL"].min(),
    "maximum_end_local": noaa["END_PARSED_LOCAL"].max(),
    "negative_duration_records": (noaa["DURATION_HOURS"] < 0).sum(),
    "timezone_values": noaa["CZ_TIMEZONE"].nunique(dropna=True),
}])
display(temporal_summary)

coordinate_summary = pd.DataFrame([{
    "missing_begin_coordinates": (noaa["BEGIN_LAT_NUM"].isna() | noaa["BEGIN_LON_NUM"].isna()).sum(),
    "invalid_begin_latitude": (~noaa["BEGIN_LAT_NUM"].between(-90, 90) & noaa["BEGIN_LAT_NUM"].notna()).sum(),
    "invalid_begin_longitude": (~noaa["BEGIN_LON_NUM"].between(-180, 180) & noaa["BEGIN_LON_NUM"].notna()).sum(),
}])
display(coordinate_summary)

### 2.3 Event and geographic distributions

In [ ]:
event_counts = noaa["EVENT_TYPE"].value_counts(dropna=False).rename_axis("event_type").reset_index(name="count")
cz_type_counts = noaa["CZ_TYPE"].value_counts(dropna=False).rename_axis("cz_type").reset_index(name="count")
display(event_counts)
display(cz_type_counts)

fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(data=event_counts, x="count", y="event_type", ax=ax, color="steelblue")
ax.set(title="NOAA Storm Events Sample by Event Type", xlabel="Records", ylabel="Event type")
plt.tight_layout()
plt.show()

## 3. NWS Alerts

Two active-state polls were captured more than one hour apart. Both were valid empty FeatureCollections. A separately labeled recent-window response contains 238 alerts for field profiling and does not replace the incremental polls.

In [ ]:
nws_poll_1 = load_geojson(NWS_POLL_1_PATH)
nws_poll_2 = load_geojson(NWS_POLL_2_PATH)
nws_recent_page_1 = load_geojson(NWS_RECENT_PAGE_1_PATH)
nws_recent_page_2 = load_geojson(NWS_RECENT_PAGE_2_PATH)

nws_poll_summary = pd.DataFrame([
    {
        "poll": "poll_1",
        "retrieved_at_utc": nws_poll_1_metadata["retrieved_at_utc"],
        "collection_updated": nws_poll_1.get("updated"),
        "feature_count": len(nws_poll_1.get("features", [])),
        "sha256": sha256_file(NWS_POLL_1_PATH),
    },
    {
        "poll": "poll_2",
        "retrieved_at_utc": nws_poll_2_metadata["retrieved_at_utc"],
        "collection_updated": nws_poll_2.get("updated"),
        "feature_count": len(nws_poll_2.get("features", [])),
        "sha256": sha256_file(NWS_POLL_2_PATH),
    },
])
display(nws_poll_summary)

### 3.1 Recent-window pagination and schema

In [ ]:
recent_pages = [nws_recent_page_1, nws_recent_page_2]
pagination_summary = pd.DataFrame([
    {
        "page": index,
        "features": len(page.get("features", [])),
        "next_cursor_present": bool(page.get("pagination", {}).get("next")),
        "updated": page.get("updated"),
    }
    for index, page in enumerate(recent_pages, start=1)
])
display(pagination_summary)

recent_features = [
    feature
    for page in recent_pages
    for feature in page.get("features", [])
]
assert len(recent_features) >= 200, "NWS evidence does not meet the Workshop 1 record threshold."

nws_raw = pd.json_normalize(recent_features, sep=".")
print("Recent alert features:", len(recent_features))
print("Normalized columns:", nws_raw.shape[1])
display(nws_raw.head())

### 3.2 Missingness, identifiers, and content hashes

In [ ]:
nws_profile = dataframe_profile(nws_raw)
display(nws_profile)

feature_hashes = [
    hashlib.sha256(
        json.dumps(feature, sort_keys=True, separators=(",", ":"), default=str).encode("utf-8")
    ).hexdigest().upper()
    for feature in recent_features
]

nws_id_summary = pd.DataFrame([{
    "rows": len(nws_raw),
    "distinct_feature_ids": nws_raw["id"].nunique(dropna=True),
    "missing_feature_ids": nws_raw["id"].isna().sum(),
    "duplicated_feature_ids": nws_raw["id"].duplicated(keep=False).sum(),
    "distinct_property_ids": nws_raw["properties.id"].nunique(dropna=True),
    "distinct_content_hashes": pd.Series(feature_hashes).nunique(),
    "null_geometry": sum(feature.get("geometry") is None for feature in recent_features),
}])
display(nws_id_summary)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
nws_null_plot = nws_profile.sort_values("null_pct", ascending=True).tail(25)
ax.barh(nws_null_plot["column"], nws_null_plot["null_pct"], color="darkorange", alpha=0.8)
ax.set(title="NWS Recent Alerts — Highest Missingness", xlabel="Null values (%)", ylabel="Column")
plt.tight_layout()
plt.show()

### 3.3 Alert distributions and update semantics

In [ ]:
event_counts_nws = nws_raw["properties.event"].value_counts(dropna=False).rename_axis("event").reset_index(name="count")
message_counts = nws_raw["properties.messageType"].value_counts(dropna=False).rename_axis("message_type").reset_index(name="count")
severity_counts = nws_raw["properties.severity"].value_counts(dropna=False).rename_axis("severity").reset_index(name="count")

display(event_counts_nws)
display(message_counts)
display(severity_counts)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=event_counts_nws, x="count", y="event", ax=axes[0], color="teal")
axes[0].set(title="NWS Alerts by Event", xlabel="Records", ylabel="Event")
sns.countplot(data=nws_raw, x="properties.messageType", hue="properties.severity", ax=axes[1])
axes[1].set(title="Message Type and Severity", xlabel="Message type", ylabel="Records")
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

### 3.4 Temporal and geographic profile

In [ ]:
nws = nws_raw.copy()
timestamp_columns = [
    "properties.sent",
    "properties.effective",
    "properties.onset",
    "properties.expires",
    "properties.ends",
]
for column in timestamp_columns:
    nws[column + "_parsed"] = pd.to_datetime(nws[column], errors="coerce", utc=True)

nws["affected_zone_count"] = nws["properties.affectedZones"].map(
    lambda value: len(value) if isinstance(value, list) else 0
)
nws["reference_count"] = nws["properties.references"].map(
    lambda value: len(value) if isinstance(value, list) else 0
)

temporal_rows = []
for column in timestamp_columns:
    parsed = nws[column + "_parsed"]
    temporal_rows.append({
        "field": column,
        "source_nulls": int(nws[column].isna().sum()),
        "parse_failures_non_null": int((parsed.isna() & nws[column].notna()).sum()),
        "minimum_utc": parsed.min(),
        "maximum_utc": parsed.max(),
    })
display(pd.DataFrame(temporal_rows))

geometry_types = pd.Series([
    "NULL" if feature.get("geometry") is None else feature["geometry"].get("type")
    for feature in recent_features
]).value_counts(dropna=False).rename_axis("geometry_type").reset_index(name="count")
display(geometry_types)
display(nws[["affected_zone_count", "reference_count"]].describe())

### 3.5 Incremental poll comparison

An absent alert in a later active response is not automatically a source deletion. It may have expired or otherwise become inactive. Both raw responses remain the source of truth.

In [ ]:
poll_1_features = nws_poll_1.get("features", [])
poll_2_features = nws_poll_2.get("features", [])
poll_1_by_id = {feature.get("id"): feature for feature in poll_1_features}
poll_2_by_id = {feature.get("id"): feature for feature in poll_2_features}

poll_1_ids = set(poll_1_by_id)
poll_2_ids = set(poll_2_by_id)
shared_ids = poll_1_ids & poll_2_ids
changed_ids = {
    alert_id
    for alert_id in shared_ids
    if stable_value(poll_1_by_id[alert_id]) != stable_value(poll_2_by_id[alert_id])
}

poll_comparison = pd.DataFrame([{
    "interval_seconds": (
        pd.Timestamp(nws_poll_2_metadata["retrieved_at_utc"])
        - pd.Timestamp(nws_poll_1_metadata["retrieved_at_utc"])
    ).total_seconds(),
    "poll_1_features": len(poll_1_features),
    "poll_2_features": len(poll_2_features),
    "new_in_poll_2": len(poll_2_ids - poll_1_ids),
    "no_longer_active": len(poll_1_ids - poll_2_ids),
    "shared_changed": len(changed_ids),
    "shared_unchanged": len(shared_ids - changed_ids),
    "collection_updated_changed": nws_poll_1.get("updated") != nws_poll_2.get("updated"),
    "document_hash_changed": sha256_file(NWS_POLL_1_PATH) != sha256_file(NWS_POLL_2_PATH),
}])
display(poll_comparison)

## 4. GHCN-Daily / NCEI

In [ ]:
ghcn_raw = pd.read_csv(GHCN_SAMPLE_PATH, parse_dates=["DATE"])
station_observation_payload = load_geojson(STATION_OBSERVATIONS_PATH)
station_features = station_observation_payload["features"]
station_observations = pd.json_normalize([feature["properties"] for feature in station_features])
print(f"GHCN-Daily rows: {len(ghcn_raw):,}; NWS station-observation features: {len(station_features):,}")

GHCN rows: 365; NWS observation features: 200


In [ ]:
ghcn_profile = pd.DataFrame([
    {"check": "rows", "result": len(ghcn_raw)},
    {"check": "columns", "result": len(ghcn_raw.columns)},
    {"check": "date_start", "result": ghcn_raw["DATE"].min().date().isoformat()},
    {"check": "date_end", "result": ghcn_raw["DATE"].max().date().isoformat()},
    {"check": "unique_dates", "result": ghcn_raw["DATE"].nunique()},
    {"check": "duplicate station-date keys", "result": ghcn_raw.duplicated(["STATION", "DATE"]).sum()},
    {"check": "stations", "result": ghcn_raw["STATION"].nunique()},
    {"check": "missing coordinate rows", "result": ghcn_raw[["LATITUDE", "LONGITUDE"]].isna().any(axis=1).sum()},
])
print(ghcn_profile.to_string(index=False))

                      check     result
                       rows        365
                    columns         12
                 date_start 2025-01-01
                   date_end 2025-12-31
               unique_dates        365
duplicate station-date keys          0
                   stations          1
    missing coordinate rows          0


In [ ]:
ghcn_measurements = []
for field in ["PRCP", "TMAX", "TMIN"]:
    values = pd.to_numeric(ghcn_raw[field], errors="coerce")
    attrs = ghcn_raw[f"{field}_ATTRIBUTES"].fillna("").astype(str).str.split(",", expand=True)
    ghcn_measurements.append({
        "field": field,
        "rows": len(ghcn_raw),
        "null_values": int(values.isna().sum()),
        "minimum": values.min(),
        "maximum": values.max(),
        "mean": round(values.mean(), 3),
        "measurement_flags": attrs[0].replace("", "(blank)").value_counts().to_dict(),
        "quality_flags": attrs[1].replace("", "(blank)").value_counts().to_dict(),
        "source_flags": attrs[2].replace("", "(blank)").value_counts().to_dict(),
    })
print(pd.DataFrame(ghcn_measurements).to_string(index=False))

field  rows  null_values  minimum  maximum   mean         measurement_flags    quality_flags         source_flags
 PRCP   365            0      0.0     36.3  1.072 {'(blank)': 318, 'T': 47} {'(blank)': 365} {'W': 238, '1': 127}
 TMAX   365            0    -12.1     37.8 19.655          {'(blank)': 365} {'(blank)': 365} {'W': 238, '1': 127}
 TMIN   365            0    -25.5     23.3  3.618          {'(blank)': 365} {'(blank)': 365} {'W': 238, '1': 127}


The NCEI request asked for metric output. These ranges describe this one-station, one-year subset only. Preserve the attribute columns: PRCP has trace flags (`T`) on records whose numeric value is `0.0`, so dropping flags would conflate trace precipitation with ordinary zero.

### 4.1 NWS station observations

In [ ]:
station_timestamps = pd.to_datetime(station_observations["timestamp"], utc=True, errors="coerce")
station_coordinate_rows = [feature.get("geometry", {}).get("coordinates") for feature in station_features]
station_coordinate_pairs = [tuple(pair[:2]) for pair in station_coordinate_rows if isinstance(pair, list) and len(pair) >= 2]
station_observation_profile = pd.DataFrame([
    {"check": "station_features", "result": len(station_features)},
    {"check": "unique observation IDs", "result": pd.Series([feature.get("id") for feature in station_features]).nunique()},
    {"check": "unique station IDs", "result": station_observations["stationId"].nunique()},
    {"check": "missing/unparseable station_timestamps", "result": int(station_timestamps.isna().sum())},
    {"check": "timestamp start (UTC)", "result": station_timestamps.min().isoformat()},
    {"check": "timestamp end (UTC)", "result": station_timestamps.max().isoformat()},
    {"check": "duplicate observation station_timestamps", "result": int(station_timestamps.duplicated().sum())},
    {"check": "unique point coordinate pairs", "result": len(set(station_coordinate_pairs))},
    {"check": "response has pagination.next", "result": bool(nws.get("pagination", {}).get("next"))},
])
print(station_observation_profile.to_string(index=False))

                           check                    result
                        features                       200
          unique observation IDs                       200
              unique station IDs                         1
  missing/unparseable timestamps                         0
           timestamp start (UTC) 2026-11-02T11:30:00+00:00
             timestamp end (UTC) 2026-11-04T02:35:00+00:00
duplicate observation timestamps                         0
   unique point coordinate pairs                         2
    response has pagination.next                      True


In [ ]:
station_measurement_fields = [
    "temperature", "dewpoint", "windDirection", "windSpeed", "windGust",
    "barometricPressure", "seaLevelPressure", "visibility",
    "maxTemperatureLast24Hours", "minTemperatureLast24Hours",
    "precipitationLastHour", "precipitationLast3Hours", "precipitationLast6Hours",
    "relativeHumidity", "heatIndex", "windChill",
]
station_field_rows = []
for field in station_measurement_fields:
    entries = [feature.get("properties", {}).get(field) for feature in station_features]
    values = [entry.get("value") if isinstance(entry, dict) else None for entry in entries]
    numeric = [value for value in values if isinstance(value, (int, float)) and not isinstance(value, bool)]
    units = sorted({entry.get("unitCode") for entry in entries if isinstance(entry, dict) and entry.get("unitCode")})
    qc = pd.Series([entry.get("qualityControl") for entry in entries if isinstance(entry, dict)]).value_counts(dropna=False).to_dict()
    station_field_rows.append({
        "field": field,
        "numeric_values": len(numeric),
        "null_or_missing_values": len(station_features) - len(numeric),
        "unit_codes": ", ".join(units),
        "quality_control_counts": qc,
        "minimum": min(numeric) if numeric else None,
        "maximum": max(numeric) if numeric else None,
    })
print(pd.DataFrame(station_field_rows).to_string(index=False))

                    field  numeric_values  null_or_missing_values             unit_codes      quality_control_counts       minimum       maximum
              temperature             197                       3           wmoUnit:degC          {'V': 197, 'Z': 3}      5.000000     13.000000
                 dewpoint             198                       2           wmoUnit:degC          {'V': 198, 'Z': 2}     -1.100000      8.000000
            windDirection             174                      26 wmoUnit:degree_(angle)         {'V': 174, 'Z': 26}      0.000000    360.000000
                windSpeed             176                      24         wmoUnit:km_h-1         {'V': 176, 'Z': 24}      0.000000     50.004000
                 windGust              72                     128         wmoUnit:km_h-1 {'Z': 128, 'S': 68, 'C': 4}     27.792000     70.380000
       barometricPressure               0                     200             wmoUnit:Pa                  {'Z': 200}           NaN

In [ ]:
station_poll2_payload = load_geojson(STATION_OBSERVATIONS_POLL2_PATH)
station_poll2_features = station_poll2_payload["features"]
station_poll1_by_id = {feature.get("id"): feature for feature in station_features}
station_poll2_by_id = {feature.get("id"): feature for feature in station_poll2_features}
poll1_observation_ids = set(station_poll1_by_id)
poll2_observation_ids = set(station_poll2_by_id)
shared_observation_ids = poll1_observation_ids & poll2_observation_ids
changed_observation_ids = {
    observation_id for observation_id in shared_observation_ids
    if stable_value(station_poll1_by_id[observation_id]) != stable_value(station_poll2_by_id[observation_id])
}
station_poll_interval_seconds = (
    pd.Timestamp(station_observations_poll2_metadata["retrievedAtUtc"])
    - pd.Timestamp(station_observations_metadata["retrievedAtUtc"])
).total_seconds()
station_poll_comparison = pd.DataFrame([{
    "interval_seconds": station_poll_interval_seconds,
    "poll_1_features": len(station_features),
    "poll_2_features": len(station_poll2_features),
    "shared_ids": len(shared_observation_ids),
    "new_in_poll_2": len(poll2_observation_ids - poll1_observation_ids),
    "not_in_poll_2_page": len(poll1_observation_ids - poll2_observation_ids),
    "changed_shared_payloads": len(changed_observation_ids),
    "unchanged_shared_payloads": len(shared_observation_ids - changed_observation_ids),
    "poll_2_next_page_present": bool(station_poll2_payload.get("pagination", {}).get("next")),
}])
display(station_poll_comparison)
for observation_id in changed_observation_ids:
    before = station_poll1_by_id[observation_id]["properties"]
    after = station_poll2_by_id[observation_id]["properties"]
    changed_fields = [key for key in sorted(set(before) | set(after)) if before.get(key) != after.get(key)]
    print("Changed observation:", observation_id)
    print("Changed properties:", changed_fields)


### Pablo source-profile limits

- The GHCN profile is one station and 2025 only; it does not establish regional coverage or historical volume.
- Both NWS station samples are bounded first pages of 200 observations and include `pagination.next`. Their timestamp windows overlap for some IDs; one shared ID changed payload between polls. Two polls do not establish broader retention or the endpoint-wide revision policy.
- Null measurements are reported as observed availability, not automatically classified as defects; some elements may be conditional and QC codes need interpretation.
- Denver is a candidate crosswalk only. The team must choose the final region and aggregation rule together.
- The second NWS station-observation poll is preserved; 104 records entered and 104 left the bounded 200-feature page, and one overlapping observation changed at the same ID and timestamp. This is an observed payload revision, not proof of the service-wide revision policy.

## 5. Evidence-backed quality observations

The following table is generated from the captured samples. These observations are candidates for the Workshop 1 quality findings; interpretation must distinguish expected source semantics from actual defects.

In [ ]:
quality_observations = pd.DataFrame([
    {
        "source": "NOAA Storm Events",
        "observation": "Missing begin coordinates",
        "affected_records": int((noaa["BEGIN_LAT_NUM"].isna() | noaa["BEGIN_LON_NUM"].isna()).sum()),
        "interpretation_status": "Requires CZ_TYPE and zone/county analysis",
    },
    {
        "source": "NOAA Storm Events",
        "observation": "Timestamp parse failures",
        "affected_records": int(noaa["BEGIN_PARSED_LOCAL"].isna().sum() + noaa["END_PARSED_LOCAL"].isna().sum()),
        "interpretation_status": "Verify examples before classifying",
    },
    {
        "source": "NWS Alerts",
        "observation": "Null geometry",
        "affected_records": int(sum(feature.get("geometry") is None for feature in recent_features)),
        "interpretation_status": "Check affectedZones and geocode first",
    },
    {
        "source": "NWS Alerts",
        "observation": "Null ends timestamp",
        "affected_records": int(nws["properties.ends"].isna().sum()),
        "interpretation_status": "May be valid CAP semantics",
    },
    {
        "source": "NWS Alerts",
        "observation": "Null instruction",
        "affected_records": int(nws["properties.instruction"].isna().sum()),
        "interpretation_status": "Assess by event and message type",
    },
])
station_quality_candidates = []
for field in ["barometricPressure", "maxTemperatureLast24Hours", "minTemperatureLast24Hours", "precipitationLast3Hours", "precipitationLastHour", "windGust"]:
    field_summary = next(row for row in station_field_rows if row["field"] == field)
    station_quality_candidates.append({"source":"NWS Station Observations", "observation":f"Null {field}.value in captured page", "affected_records":field_summary["null_or_missing_values"], "interpretation_status":"Sample-level missingness; assess optionality and QC before calling a defect"})
station_quality_candidates.extend([
    {"source": "GHCN-Daily", "observation": "PRCP trace flag with numeric zero", "affected_records": int(((ghcn_raw["PRCP"] == 0) & ghcn_raw["PRCP_ATTRIBUTES"].fillna("").str.startswith("T")).sum()), "interpretation_status": "Preserve trace flag; not equivalent to ordinary zero"},
    {"source": "NWS Station Observations", "observation": "Observation point varies within one station page", "affected_records": len(set(station_coordinate_pairs)), "interpretation_status": "Two coordinate pairs observed; inspect provider/station metadata before treating station location as fixed"},
    {"source": "NWS Station Observations", "observation": "Observation intervals vary", "affected_records": int((pd.Series(station_timestamps.sort_values().diff().dropna().dt.total_seconds()) != 300).sum()), "interpretation_status": "Intervals outside five minutes are observed cadence variation, not automatically a defect"},
])
quality_observations = pd.concat([quality_observations, pd.DataFrame(station_quality_candidates)], ignore_index=True)
display(quality_observations)

## 6. Preliminary source-to-canonical mapping

This is a profiling aid, not the final accepted data contract. The GHCN-Daily and station-observation mappings below preserve daily measurements and source-specific units/QC without treating every station reading as a discrete hazard event.

In [ ]:
preliminary_mapping = pd.DataFrame([
    ["source_record_id", "NOAA", "EVENT_ID", "Verify uniqueness across selected years"],
    ["source_record_id", "NWS Alerts", "id / properties.id", "Compare values and update chains"],
    ["event_type", "NOAA", "EVENT_TYPE", "Controlled Storm Data event type"],
    ["event_type", "NWS Alerts", "properties.event", "Alert product name"],
    ["event_time", "NOAA", "BEGIN_DATE_TIME + CZ_TIMEZONE", "UTC conversion rule pending"],
    ["event_time", "NWS Alerts", "properties.effective", "RFC 3339 timestamp"],
    ["end_time", "NOAA", "END_DATE_TIME + CZ_TIMEZONE", "UTC conversion rule pending"],
    ["end_time", "NWS Alerts", "properties.ends / expires", "Semantic priority pending"],
    ["geometry", "NOAA", "BEGIN/END_LAT/LON + CZ identifiers", "Point may be absent for zone events"],
    ["geometry", "NWS Alerts", "geometry + affectedZones + geocode", "Null polygon can still have zones"],
    ["severity", "NOAA", "hazard-specific magnitude and impacts", "No universal source scale"],
    ["severity", "NWS Alerts", "properties.severity", "CAP categorical severity"],
    ["raw_reference", "NOAA", "archive filename + EVENT_ID", "Include archive checksum"],
    ["raw_reference", "NWS Alerts", "response path + feature id", "Include response checksum"],
    ["station_id", "GHCN-Daily", "STATION", "Preserve the original station identifier"],
    ["observation_date", "GHCN-Daily", "DATE", "Daily date; do not invent an observation time"],
    ["measurement_type/value", "GHCN-Daily", "PRCP/TMAX/TMIN + attributes", "Keep element, units, and M/Q/S flags together; not a discrete event by default"],
    ["station_id", "NWS Station Observations", "properties.stationId", "Preserve station identifier; test crosswalk to GHCN separately"],
    ["source_record_id", "NWS Station Observations", "feature.id / properties.@id", "ID includes timestamp; overlapping poll showed same-ID payload revision, so compare content hash as well"],
    ["event_time", "NWS Station Observations", "properties.timestamp", "Parse offset-aware timestamp to UTC; retain retrieval time separately"],
    ["measurement value/unit/QC", "NWS Station Observations", "properties.<element>.value/unitCode/qualityControl", "Preserve raw property, value, unit and QC; poll comparison demonstrated QC/value revision"],
    ["geometry", "NWS Station Observations", "feature.geometry.coordinates", "GeoJSON point; compare station coordinate variation before region mapping"],
    ["raw_reference", "GHCN-Daily and NWS Station Observations", "sample path + checksum", "Preserve raw response and its evidence metadata"],
], columns=["canonical_field", "source", "source_field", "note"])
display(preliminary_mapping)

## 7. Limitations and next steps

### Current limitations

- Oklahoma/county aggregation and Denver are candidate scopes only; one Denver station does not validate regional coverage.
- The NOAA sample covers one complete year and does not prove the proposed 2010–2025 historical window.
- Both NWS Alerts active polls were empty; station-observation polls did show one changed overlapping payload, but do not establish a service-wide revision policy.
- The recent NWS collection supports field profiling but is not a long-term archive.
- Null source values are observations, not automatically defects.
- No cleaning, imputation, outlier removal, or canonical normalization is performed here.

### Required next steps

1. Have Kevin review the station poll comparison and confirm the provisional content-hash revision strategy.
2. Confirm the region and historical window using coverage evidence from all four sources.
3. Review candidate data-quality findings and attach source examples.
4. Investigate five candidate same-event pairs across sources.
5. Complete the canonical mapping, contracts, ER diagram, and preliminary index proposal.

## 8. Execution summary

Run all cells from top to bottom before submission. Save outputs only after every assertion passes and the results have been reviewed against the registered evidence.

In [ ]:
execution_summary = pd.DataFrame([
    {"source": "NOAA Storm Events", "records": len(noaa_raw), "section_implemented": True},
    {"source": "NWS Alerts active poll 1", "records": len(poll_1_features), "section_implemented": True},
    {"source": "NWS Alerts active poll 2", "records": len(poll_2_features), "section_implemented": True},
    {"source": "NWS Alerts recent window", "records": len(recent_features), "section_implemented": True},
    {"source": "GHCN-Daily", "records": len(ghcn_raw), "section_implemented": True},
    {"source": "NWS Station Observations poll 1", "records": len(station_features), "section_implemented": True},
    {"source": "NWS Station Observations poll 2", "records": len(station_poll2_features), "section_implemented": True},
])
display(execution_summary)

print("Initial profiles and preserved samples are present for all four sources; joint review and shared decisions remain pending.")
print("Execute the full notebook top to bottom after all evidence checks and obtain review before submission.")